# Part 2: Tensors

A tensor is an array with a **shape**, a **dtype** and a **device**. Everything in PyTorch is made of them.

In [1]:
import numpy as np
import torch

print("PyTorch", torch.__version__)
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(a)
print("shape:", a.shape, " dtype:", a.dtype, " device:", a.device)

PyTorch 2.15.0+cpu
tensor([[1., 2., 3.],
        [4., 5., 6.]])
shape: torch.Size([2, 3])  dtype: torch.float32  device: cpu


## Making tensors

In [2]:
print(torch.zeros(2, 3))
print(torch.ones(2))
print(torch.arange(6).reshape(2, 3))
torch.manual_seed(0)
print(torch.randn(2, 3))
print("integers get int64:", torch.tensor([1, 2, 3]).dtype)
print("decimals get float32:", torch.tensor([1.5, 2.5]).dtype)

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([1., 1.])
tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986]])
integers get int64: torch.int64
decimals get float32: torch.float32


## Indexing, slicing and reshaping

In [3]:
m = torch.arange(12).reshape(3, 4)
print(m)
print("one row:", m[1])
print("one column:", m[:, 2])
print("one value:", m[1, 2].item())
print("rows where first column > 3:\n", m[m[:, 0] > 3])
print("flattened:", m.reshape(-1).shape, " as 4x3:", m.reshape(4, 3).shape)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
one row: tensor([4, 5, 6, 7])
one column: tensor([ 2,  6, 10])
one value: 6
rows where first column > 3:
 tensor([[ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
flattened: torch.Size([12])  as 4x3: torch.Size([4, 3])


## Arithmetic and broadcasting

In [4]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("x * 2:\n", x * 2)
print("x + a row of three (broadcast down the rows):\n", x + torch.tensor([10.0, 20.0, 30.0]))
print("column means:", x.mean(dim=0), " row means:", x.mean(dim=1))
w = torch.tensor([[1.0], [0.5], [0.0]])
print("matrix product x @ w:", (x @ w).shape, "\n", x @ w)

x * 2:
 tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]])
x + a row of three (broadcast down the rows):
 tensor([[11., 22., 33.],
        [14., 25., 36.]])
column means: tensor([2.5000, 3.5000, 4.5000])  row means: tensor([2., 5.])
matrix product x @ w: torch.Size([2, 1]) 
 tensor([[2.0000],
        [6.5000]])


## Shapes that almost match

A shape of `(4,)` and a shape of `(4, 1)` hold the same four numbers, but they are different things, and PyTorch will broadcast them into a table.

In [5]:
p = torch.tensor([0.9, 0.2, 0.7, 0.1])
y = torch.tensor([[1.0], [0.0], [1.0], [0.0]])
print("p:", p.shape, " y:", y.shape)
print("p - y has shape:", (p - y).shape, " (a 4x4 table, not four differences)")
print("fixed:", (p - y.squeeze(1)).shape)

p: torch.Size([4])  y: torch.Size([4, 1])
p - y has shape: torch.Size([4, 4])  (a 4x4 table, not four differences)
fixed: torch.Size([4])


## Data types

Models expect `float32`. A NumPy array of decimals is `float64`, so a conversion is needed.

In [6]:
arr = np.array([0.5, 1.5, 2.5])
t = torch.from_numpy(arr)
print("from NumPy:", t.dtype)
print("converted:", t.float().dtype, " or ", torch.tensor(arr, dtype=torch.float32).dtype)
layer = torch.nn.Linear(3, 1)
try:
    layer(t)
except RuntimeError as err:
    print("error:", err)

from NumPy: torch.float64
converted: torch.float32  or  torch.float32
error: mat1 and mat2 must have the same dtype, but got Double and Float


## Shared memory with NumPy

`torch.from_numpy` does not copy. Change one and you change the other.

In [7]:
arr = np.array([1.0, 2.0, 3.0])
shared = torch.from_numpy(arr)
shared[0] = 99.0
print("the NumPy array changed too:", arr)
independent = torch.tensor(arr)       # torch.tensor copies
independent[1] = -1.0
print("the original did not change:", arr)

the NumPy array changed too: [99.  2.  3.]
the original did not change: [99.  2.  3.]


## Devices

A tensor lives on one device. Tensors on different devices cannot be combined. Most laptops have only the CPU, so the code below picks whatever is available.

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("using:", device)
z = torch.ones(3, device=device)
print(z.device, " back to the CPU for NumPy:", z.cpu().numpy())

using: cpu
cpu  back to the CPU for NumPy: [1. 1. 1.]


## Your turn

1. Make a `(5, 3)` tensor of random numbers and print the mean of each column.
2. Normalize it: subtract the column means and divide by the column standard deviations using broadcasting. Check that the new column means are close to zero.
3. Convert a Pandas column to a `float32` tensor with `torch.tensor(df['col'].to_numpy(), dtype=torch.float32)`.